# Building and Running a Pipeline with NodeML

This notebook walks through the core workflow of the **Time Series Unified Toolbox (NodeML)**:

1. **Discover** available nodes via the registry
2. **Configure** each node (data sources, transforms, models, metrics)
3. **Assemble** nodes and edges into a `Pipeline`
4. **Run** training, inference, and evaluation through a `SmartRunner`

## 1. Setup and Node Discovery

NodeML auto-discovers all built-in component nodes at import time. The global `NODE_REGISTRY` holds every registered node type. We configure logging to see what happens under the hood.

In [ ]:
%matplotlib ipympl
import sys

from nodeml import NODE_REGISTRY
from nodeml.core.common.logging import configure

configure(level="DEBUG", stream=sys.stdout, fmt="text")

In [ ]:
NODE_REGISTRY.list()

## 2. Configuring Data Source Nodes

Each node type has a **Config** class (retrievable from the registry) that bundles:
- **`hyperparameters`** -- tuneable parameters for hyperparameter search
- **`running_config`** -- execution-time settings (file paths, backend options, etc.)
- **`in_ports` / `out_ports`** -- the data contract (array type, shape, category)

Here we configure two `TabularCSVFetcher` sources: one for features and one for targets.

In [ ]:
csv_config_class = NODE_REGISTRY.get_node_config_class("TabularCSVFetcher")
csv_running_config_class = NODE_REGISTRY["TabularCSVFetcher"]["running_config"]

In [ ]:
csv_running_config_class

In [ ]:
data_csv_config = csv_config_class()
target_csv_config = csv_config_class()

In [ ]:
data_csv_config.running_config.csv_path = "../data/fake_batch.csv"
data_csv_config.running_config.context_path = "../data/fake_batch_context.json"

target_csv_config.running_config.csv_path = "../data/fake_target_df.csv"
target_csv_config.running_config.context_path = "../data/fake_target_context.json"

## 3. Configuring Transform Nodes

Transforms have both **hyperparameters** (e.g. `threshold`) and **running configs**. We can also override port modes -- for instance, the target missing-rate filter should only be active during training and evaluation (not inference, since we don't have ground-truth labels at inference time).

In [ ]:
from nodeml.core.pipeline.pipeline import Edge, Pipeline, PipelineConfig

In [ ]:
missing_filter_config_class = NODE_REGISTRY.get_node_config_class("MissingRateFilter")
data_missing_filter_config = missing_filter_config_class()
target_missing_filter_config = missing_filter_config_class()

target_missing_filter_config.in_ports["input"].mode = ["training", "evaluation"]
target_missing_filter_config.hyperparameters.threshold = 0.0

## 4. Assembling the Pipeline

A pipeline is a **directed acyclic graph (DAG)** defined by:
- **`nodes`** -- a dict mapping node names to `(node_type_name, config)` tuples. For nodes where you don't need custom configuration, you can pass a default config from the registry.
- **`edges`** -- a list of `Edge` objects that connect output ports of one node to input ports of another via `ports_map`.

The pipeline below implements a full ML workflow:
1. Load features and targets from CSV
2. Filter missing values, split by data category (numerical vs categorical)
3. Remove outliers (IQR), filter low-variance features, scale, one-hot encode
4. Concatenate processed features, feed into a `RandomForestRegressor`
5. Evaluate with R2 and MSE metrics

In [ ]:
nodes = {
    "data_source": ("TabularCSVFetcher", data_csv_config),
    "target_source": ("TabularCSVFetcher", target_csv_config),
    "data_missing_filter": ("MissingRateFilter", data_missing_filter_config),
    "category_splitter": (
        "DataCategoryFilter",
        NODE_REGISTRY.get_node_config_class("DataCategoryFilter")(),
    ),
    "one_hot_encoder": (
        "OneHotEncoding",
        NODE_REGISTRY.get_node_config_class("OneHotEncoding")(),
    ),
    "target_missing_filter": ("MissingRateFilter", target_missing_filter_config),
    "iqr_filter": (
        "IQROutlierFilter",
        NODE_REGISTRY.get_node_config_class("IQROutlierFilter")(),
    ),
    "variance_threshold_filter": (
        "VarianceFilter",
        NODE_REGISTRY.get_node_config_class("VarianceFilter")(),
    ),
    "standard_scaler": (
        "StandardScaler",
        NODE_REGISTRY.get_node_config_class("StandardScaler")(),
    ),
    "feature_concatenate": (
        "FeatureConcatenate",
        NODE_REGISTRY.get_node_config_class("FeatureConcatenate")(),
    ),
    "random_forest_regressor": (
        "RandomForestRegressor",
        NODE_REGISTRY.get_node_config_class("RandomForestRegressor")(),
    ),
    "r2": ("R2Score", NODE_REGISTRY.get_node_config_class("R2Score")()),
    "mse": ("MSE", NODE_REGISTRY.get_node_config_class("MSE")()),
    "sink": ("Sink", NODE_REGISTRY.get_node_config_class("Sink")()),
}

In [ ]:
edges = [
    Edge(
        source="data_source",
        target="data_missing_filter",
        ports_map=[("output", "input")],
    ),
    Edge(
        source="data_missing_filter",
        target="category_splitter",
        ports_map=[("output", "input")],
    ),
    Edge(
        source="one_hot_encoder",
        target="feature_concatenate",
        ports_map=[("output", "input_2")],
    ),
    Edge(
        source="category_splitter",
        target="iqr_filter",
        ports_map=[("numerical", "input")],
    ),
    Edge(
        source="category_splitter",
        target="iqr_filter",
        ports_map=[("categorical", "sliced")],
    ),
    Edge(source="target_source", target="iqr_filter", ports_map=[("output", "target")]),
    Edge(
        source="iqr_filter",
        target="variance_threshold_filter",
        ports_map=[("output", "input")],
    ),
    Edge(
        source="iqr_filter", target="one_hot_encoder", ports_map=[("sliced", "input")]
    ),
    Edge(
        source="variance_threshold_filter",
        target="standard_scaler",
        ports_map=[("output", "input")],
    ),
    Edge(
        source="standard_scaler",
        target="feature_concatenate",
        ports_map=[("output", "input_1")],
    ),
    Edge(
        source="feature_concatenate",
        target="random_forest_regressor",
        ports_map=[("output", "X")],
    ),
    Edge(
        source="iqr_filter",
        target="target_missing_filter",
        ports_map=[("target", "input")],
    ),
    Edge(
        source="target_missing_filter",
        target="random_forest_regressor",
        ports_map=[("output", "y")],
    ),
    Edge(source="random_forest_regressor", target="r2", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="r2", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="mse", ports_map=[("pred", "pred")]),
    Edge(
        source="target_missing_filter", target="mse", ports_map=[("output", "target")]
    ),
    Edge(source="random_forest_regressor", target="sink", ports_map=[("pred", "dump")]),
]

In [ ]:
pipe_conf = PipelineConfig(
    nodes=nodes,
    edges=edges,
)

pipe = Pipeline(config=pipe_conf)

### Visualising the graph

`pipe.render()` produces an interactive Plotly visualisation of the DAG so you can verify the topology before running anything.

In [ ]:
pipe.render()

## 5. Compiling and Running the Pipeline

Before execution, the pipeline must be **compiled**. Compilation validates the graph structure, resolves port compatibility, and prunes unconnected nodes.

A `SmartRunner` then drives the three execution phases:
- **`train()`** -- executes the full graph in training mode (fit + transform on every node)
- **`infer()`** -- re-executes in inference mode (transform only, no targets needed)
- **`evaluate()`** -- re-executes in evaluation mode and collects outputs from all metric nodes

In [ ]:
from nodeml.core.pipeline.runners.smart_runner import SmartRunner

In [ ]:
pipe.compile()
runnable = SmartRunner(pipe)

In [ ]:
runnable.train()

In [ ]:
runnable.infer()

### Inference

During inference, ports marked with `mode=["training", "evaluation"]` (like the `y` input on the model) are automatically skipped. The runner only computes what is needed to reach the sink node.

In [ ]:
eval_metrics = runnable.evaluate()

In [ ]:
eval_metrics

In [ ]:
eval_metrics["r2"]

### Evaluation

`evaluate()` runs the graph in evaluation mode and returns a dict mapping each metric node name to its computed `TabularData` output. Here we get both R2 and MSE scores.